## Stats on medication dataset

In [2]:
import pandas as pd
import numpy as np
import random

## Part 1: Diccionari de fàrmacs. Codis ATC (7 caràcters)

In [8]:
# Mapegem cada grup de diagnòstic a una llista de medicaments reals
drugs_dictionire = {
    "Anxiety disorder": [
        {'atc': 'N05BA01', 'nom': 'Diazepam', 'preu_base': 2.50},
        {'atc': 'N05BA06', 'nom': 'Lorazepam', 'preu_base': 3.10},
        {'atc': 'N05BA12', 'nom': 'Alprazolam', 'preu_base': 4.20}
    ],
    "Mood disorder": [
        {'atc': 'N06AB10', 'nom': 'Escitalopram', 'preu_base': 15.10},
        {'atc': 'N06AB06', 'nom': 'Sertralina', 'preu_base': 18.50},
        {'atc': 'N06AB03', 'nom': 'Fluoxetina', 'preu_base': 12.30}
    ],
    "Psychotic disorder": [
        {'atc': 'N05AD01', 'nom': 'Haloperidol', 'preu_base': 8.50},
        {'atc': 'N05AH02', 'nom': 'Clozapina', 'preu_base': 42.10},
        {'atc': 'N05AH03', 'nom': 'Olanzapina', 'preu_base': 35.20},
        {'atc': 'N05AH04', 'nom': 'Quetiapina', 'preu_base': 32.40},
        {'atc': 'N05AX08', 'nom': 'Risperidona', 'preu_base': 28.50},
        {'atc': 'N05AX12', 'nom': 'Aripiprazol', 'preu_base': 55.00}
    ],
    "Substance use disorder": [
        {'atc': 'N07BB01', 'nom': 'Disulfiram', 'preu_base': 22.30},
        {'atc': 'N05BA06', 'nom': 'Lorazepam', 'preu_base': 3.10}
    ]
}

## Part 2: Càrrega de dades i selecció de la mostra

In [10]:
path_diag = r'C:\Users\LRAJAG\Desktop\Data_sintetica\csv\5_Diagnose\diagnostics_psiquiatrics.csv'
df_diag = pd.read_csv(path_diag)

proportions = {
    "Anxiety disorder": 0.3138,
    "Mood disorder": 0.1883,
    "Substance use disorder": 0.1359,
    "Psychotic disorder": 0.035
}

total_prob = sum(proportions.values())
df_casos_llista = []

print("Seleccionant pacients per grups...")

for grup, prob in proportions.items():
    n_mostra = int((prob / total_prob) * 1000)
    
    # Usem .str.contains per si el pacient té més d'un diagnòstic (com "Anxiety disorder, Mood disorder")
    filtre_grup = df_diag[df_diag['nom_diagnostic_complet'].str.contains(grup, na=False)]
    
    if len(filtre_grup) >= n_mostra:
        mostra_grup = filtre_grup.sample(n_mostra, random_state=42)
        # Guardem el grup "principal" per assignar el fàrmac després
        mostra_grup['grup_per_farmac'] = grup 
        df_casos_llista.append(mostra_grup)
    else:
        print(f"Alerta: No hi ha prou casos de {grup}")

df_casos = pd.concat(df_casos_llista).sample(frac=1).reset_index(drop=True)

Seleccionant pacients per grups...


## Part 3: Generació de dispensacions

In [15]:
mesos = [f"{a}{m:02d}" for a in range(2010, 2017) for m in range(1, 13)]
dades_dispensacio = []

print(f"Generant dispensacions per a 1.000 pacients...")

for _, pacient in df_casos.iterrows():
    id_p = pacient['id']
    grup = pacient['nom_diagnostic_complet']
    
    # Si el grup té medicaments assignats al nostre diccionari
    if grup in drugs_dictionire:
        # Assignem UN medicament fix per a tota la durada de l'estudi (tractament crònic)
        farmac = random.choice(drugs_dictionire[grup])
        
        for mes in mesos:
            # Simulem un 85% d'adherència (no tots els mesos van a la farmàcia)
            if random.random() < 0.85:
                # Variació petita de preu (inflació/canvis de mercat simulats)
                pvp = round(farmac['preu_base'] * random.uniform(0.98, 1.05), 2)
                # Aportació típica (10% per a molts tractaments crònics)
                aportacio = round(pvp * 0.10, 2)
                
                dades_dispensacio.append({
                    'id': id_p,
                    'mes_dispensacio': mes,
                    'codi_atc': farmac['atc'],
                    'nom_farmac': farmac['nom'],
                    'num_envasos': 1,
                    'dies_tractament': 30,
                    'preu_pvp': pvp,
                    'aportacio_usuari': aportacio
                })

Generant dispensacions per a 1.000 pacients...


## Part 4: Creació del fitxer i estadístiques

In [16]:
df_farmacia = pd.DataFrame(dades_dispensacio)

# Guardar el CSV
path_sortida = r'C:\Users\LRAJAG\Desktop\Data_sintetica\csv\3_Medication\drugs_dispensations.csv'
df_farmacia.to_csv(path_sortida, index=False)

print(f"\n✅ Procés finalitzat!")
print(f"Total de registres generats: {len(df_farmacia):,}")
print(f"Mida estimada del fitxer: {round(len(df_farmacia)*80/1024/1024, 2)} MB (Apte per a GitHub)")
print("\nMostra de les primeres 5 files:")
print(df_farmacia.head())


✅ Procés finalitzat!
Total de registres generats: 54,754
Mida estimada del fitxer: 4.18 MB (Apte per a GitHub)

Mostra de les primeres 5 files:
      id mes_dispensacio codi_atc  nom_farmac  num_envasos  dies_tractament  \
0  42523          201002  N06AB03  Fluoxetina            1               30   
1  42523          201003  N06AB03  Fluoxetina            1               30   
2  42523          201004  N06AB03  Fluoxetina            1               30   
3  42523          201005  N06AB03  Fluoxetina            1               30   
4  42523          201006  N06AB03  Fluoxetina            1               30   

   preu_pvp  aportacio_usuari  
0     12.42              1.24  
1     12.72              1.27  
2     12.10              1.21  
3     12.09              1.21  
4     12.10              1.21  
